# 03 · Detect · Review · Govern

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AreevAI/dejadb/blob/main/examples/colab/03_detect_review_govern.ipynb)

*Segments: "Detecting repeated tool failures and contradictory memories" and
"Reviewing and approving agent-generated improvements."*

A customer-success desk after a busy quarter: a CRM export that keeps timing
out, an ordinarily-flaky product search, a contract whose status two systems
disagree on, and a promotion that expired a month ago. Waiser's detection layer
is **deterministic** — 11 versioned analyzers, threshold-gated, precision
test-enforced ≥ 0.90, no API key — and everything it finds enters a four-gate
lifecycle:

| Gate | What it enforces |
|---|---|
| **Propose** | structured recommendations with bounded evidence — never free prose |
| **Review** | mandatory `because` reason; separation of duties; no self-approval |
| **Apply** | destructive payloads double-gated; every apply records its inverse |
| **Verify** | re-measured at 1d/7d/30d — regressions file their own revert |

In [1]:
# dejadb 1.0.5 is on PyPI; `dejadb.helpers` ships inside the wheel.
%pip install -q "dejadb>=1.0.5" matplotlib

from dejadb.helpers import *
import dejadb, json, pathlib
print("dejadb", dejadb.__version__)

dejadb 1.0.3


## The mess, as the agent recorded it

In [2]:
db = fresh("success-desk.db", ns="accounts", actor="user:reviewer")

for i in range(1, 5):                                     # a real pattern: 4 of 5 failed
    db.record_tool_call("crm_export", f"timeout after 30s (run {i})",
                        is_error=True, thread="q2")
db.record_tool_call("crm_export", '{"rows": 1240}', thread="q2")

for i in range(1, 4):                                     # ordinary flakiness: 3 of 10
    db.record_tool_call("product_search", f"429 rate limited (q {i})",
                        is_error=True, thread="q2")
for i in range(7):
    db.record_tool_call("product_search", json.dumps({"hits": i}), thread="q2")

db.add_fact("contract-9107", "status", "in_review")       # two systems disagree
db.add_fact("contract-9107", "status", "signed")

db.add("fact", json.dumps({"subject": "promo-Q2", "relation": "discount_code",
                           "object": "SAVE20",            # expired a month ago
                           "vt": int(__import__("time").time()*1000) - 30*24*3600*1000}))
print("seeded")

seeded


## Detect — one sweep, and note what's *not* flagged

In [3]:
run = json.loads(db.waiser_run(full_sweep=True))
pending = show_recs(db)

print()
print("product_search (30% errors) flagged?",
      any("product_search" in r["summary"] for r in pending),
      "— below the evidence threshold. Signal, not noise.")

  [medium] [reversible ] "contract-9107" holds 2 live values for functional relation "status"
  [low   ] [DESTRUCTIVE] Expire "promo-Q2": past its declared valid_to (30d ago)
  [medium] [reversible ] Tool "crm_export" failed 4 times (80% of calls): timeout after #s (run #)

product_search (30% errors) flagged? False — below the evidence threshold. Signal, not noise.


## Review — with judgment, never a rubber stamp

Approve two. The third wants to **delete** the expired promo — destructive, so
the apply gate would demand an explicit `allow_destructive=True` anyway — and
here human judgment overrides entirely:

In [4]:
by = {r["analyzer"].split(".")[1].split("/")[0]: r for r in pending}

db.apply_recommendation(by["tool_failure"]["hash"],
    because="export timeouts confirmed — fall back to yesterday's cached export")
db.apply_recommendation(by["contradiction_sweep"]["hash"],
    because="legal confirmed the countersigned copy — 'signed' is correct")
db.dismiss_recommendation(by["staleness"]["hash"],
    "promo under legal hold — retain until the 2027 audit closes")

print("the contradiction was superseded, not deleted:")
for v in json.loads(db.history("contract-9107", "status")):
    print(f"  [{'superseded' if v['superseded_by'] else 'LIVE      '}] {v['object']!r}")

the contradiction was superseded, not deleted:
  [LIVE      ] 'signed'
  [superseded] 'in_review'


## Govern — autonomy is granted, scoped, and revocable

Nothing self-applies by default. Autonomy comes only from a **policy file** —
per analyzer, per target, severity-capped (the console and MCP surface
deliberately cannot grant it). Typical team policy: structural dedup may
self-apply at low severity; everything else queues:

In [5]:
pathlib.Path("team-policy.json").write_text(json.dumps({
    "auto_apply_enabled": True,
    "auto_apply": [{"analyzer": "waiser.duplicate_sweep",
                    "targets": ["memory"], "max_severity": "low"}],
    "deny": [], "severity_floors": {}, "telemetry": "aggregate"}))

db.add_fact("globex", "account_owner", "dana@globex.example")   # two ingest pipelines,
db.add_fact("globex", "account_owner", "dana@globex.example")   # one classic duplicate

run = json.loads(db.waiser_run(full_sweep=True, policy="team-policy.json"))
print(f"auto_applied={run['auto_applied']} — dedup self-applied under policy; "
      "logged and reversible like everything else")

auto_applied=2 — dedup self-applied under policy; logged and reversible like everything else


The trail answers the auditor's question — *who changed what the agent knows,
when, and why:*

In [6]:
audit(db)

  [applied    ] "contract-9107" holds 2 live values for functional relation "statu
  [applied    ] Consolidate 2 exact-duplicate grains for "contract-9107"
  [applied    ] Tool "crm_export" failed 4 times (80% of calls): timeout after #s 
  [applied    ] Consolidate 2 exact-duplicate grains for "globex"
  [rejected   ] Expire "promo-Q2": past its declared valid_to (30d ago)


## Beyond the notebook

- **Review in a browser** — `deja ui --db success-desk.db` serves a local
  console with this same queue (reasons required; writes token-gated).
- **Review from CI** — `deja waiser list --fail-on high` exits non-zero while
  high-severity findings await review: a build gate for agent knowledge.
- **Separation of duties** — write, review, and apply are separate scopes; the
  proposer can't approve their own change.

When an *approved* lesson turns out wrong → `02_the_wrong_lesson.ipynb`.